# The Kernel Trick

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/svm/02-kernel-trick

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor'] = '#1a1d27'
plt.rcParams['text.color'] = 'white'
plt.rcParams['axes.labelcolor'] = '#94a3b8'
plt.rcParams['xtick.color'] = '#94a3b8'
plt.rcParams['ytick.color'] = '#94a3b8'
plt.rcParams['axes.edgecolor'] = '#2e3347'

## Intuition — curved boundaries without the cost

A linear SVM only draws hyperplanes, but many datasets aren't linearly separable. The fix: map the
data into a higher-dimensional space where it *becomes* separable, then draw the hyperplane there. The
**kernel trick** is the genius part — a **kernel function** `K(x, x') = φ(x)·φ(x')` computes the inner
product in that high-dimensional space **without ever constructing the high-dimensional vectors**. So
you get the power of (sometimes infinite-dimensional) feature maps at the cost of a cheap function
call. The **RBF kernel** is the popular default; its `gamma` controls how local the influence is. We
verify the trick numerically and validate against `sklearn`.

## Kernel Functions

In [ ]:
x1 = np.array([0, 0])
x2 = np.array([1, 1])

def linear_kernel(x1, x2): return x1 @ x2
def poly_kernel(x1, x2, c=1, d=2): return (x1 @ x2 + c)**d
def rbf_kernel(x1, x2, gamma=1): return np.exp(-gamma * np.linalg.norm(x1 - x2)**2)

print(f'Linear: {linear_kernel(x1, x2)}')
print(f'Poly (d=2): {poly_kernel(x1, x2)}')
print(f'RBF (γ=1): {rbf_kernel(x1, x2):.4f}')
print(f'RBF (γ=0.1): {rbf_kernel(x1, x2, gamma=0.1):.4f}')

**What to notice:** a kernel is a **similarity** measure between two points. The linear kernel is
just the dot product; the RBF kernel `exp(−γ‖x−x'‖²)` is 1 for identical points and decays with
distance. Every kernel secretly corresponds to an inner product in some feature space — the next cells
make that concrete.

## Linear vs RBF: Non-linearly separable data

In [ ]:
from sklearn.datasets import make_moons
from sklearn.svm import SVC

X, y = make_moons(n_samples=200, noise=0.15, random_state=42)
xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
for ax, kernel in zip(axes, ['linear', 'poly', 'rbf']):
    svm = SVC(kernel=kernel, C=1.0, gamma='scale')
    svm.fit(X, y)
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(X[y == 0, 0], X[y == 0, 1], c='#f43f5e', s=15, alpha=0.7)
    ax.scatter(X[y == 1, 0], X[y == 1, 1], c='#818cf8', s=15, alpha=0.7)
    ax.set_title(f'{kernel.capitalize()} Kernel', color='white')
plt.suptitle('SVM with Different Kernels', color='white', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

**What to notice:** on the two-moons data the **linear** kernel fails (no line separates the
interleaved crescents), while the **poly** and **RBF** kernels carve the correct **curved** boundaries.
Same SVM algorithm, same data — only the kernel changed. That's the kernel trick turning a linear
classifier non-linear.

## The kernel trick: implicit feature maps

A kernel $K(x, x')$ computes a dot product in a high-dimensional space **without ever building it**. The RBF kernel corresponds to an infinite-dimensional space.

In [ ]:
# Explicit degree-2 map vs the polynomial kernel give the same dot product
def phi(x):  # map (a, b) -> (a^2, b^2, sqrt(2) a b, sqrt(2) a, sqrt(2) b, 1)
    a, b = x
    return np.array([a**2, b**2, np.sqrt(2)*a*b, np.sqrt(2)*a, np.sqrt(2)*b, 1])

x1, x2 = np.array([1.0, 2.0]), np.array([3.0, -1.0])
print('explicit  phi(x1).phi(x2):', phi(x1) @ phi(x2))
print('kernel    (x1.x2 + 1)^2 :', (x1 @ x2 + 1)**2)

**What to notice:** this is the kernel trick in one line — the explicit degree-2 feature map's dot
product `φ(x₁)·φ(x₂)` equals `(x₁·x₂ + 1)²` **exactly**. The kernel computed a 6-dimensional inner
product using only the 2-D inputs and a squaring. For the RBF kernel the implicit feature space is
*infinite*-dimensional, yet the kernel is still a cheap formula — you never pay for the dimensions.

In [ ]:
# Mercer's condition: a valid kernel has a symmetric POSITIVE SEMI-DEFINITE
# Gram matrix on every finite point set (all eigenvalues >= 0).
import numpy as np
rng = np.random.default_rng(0)
P = rng.normal(size=(6, 2))

def gram(K):
    return np.array([[K(a, b) for b in P] for a in P])

lin = lambda a, b: a @ b
poly = lambda a, b: (a @ b + 1) ** 2
rbf = lambda a, b: np.exp(-0.5 * np.sum((a - b) ** 2))
sig = lambda a, b: np.tanh(1.0 * (a @ b) + 1.0)   # NOT generally a valid kernel

for name, K in [('linear', lin), ('poly d=2', poly), ('rbf', rbf), ('sigmoid', sig)]:
    G = gram(K)
    eig = np.linalg.eigvalsh((G + G.T) / 2)        # symmetric part
    psd = np.all(eig >= -1e-8)
    print(f'{name:9s}: min eigenvalue = {eig.min():+.3f}  -> PSD (valid kernel)? {psd}')


**What to notice:** **Mercer's condition** tells you which functions are valid kernels: the **Gram
matrix** (all pairwise kernel values) must be **symmetric positive semi-definite**. That guarantees the
kernel corresponds to a real inner-product space, which is what makes the SVM's convex optimization
well-posed.

## gamma controls RBF reach

Small `gamma` = smooth, far-reaching influence; large `gamma` = tight, wiggly boundaries that can overfit.

In [ ]:
from sklearn.svm import SVC
from sklearn.datasets import make_moons

X, y = make_moons(n_samples=200, noise=0.2, random_state=1)
xx, yy = np.meshgrid(np.linspace(-2, 3, 200), np.linspace(-1.5, 2, 200))
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, g in zip(axes, [0.1, 1, 30]):
    svm = SVC(kernel='rbf', gamma=g, C=1).fit(X, y)
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='RdYlBu')
    ax.scatter(*X.T, c=y, cmap='RdYlBu', edgecolor='k', s=15)
    ax.set_title(f'gamma = {g}')
plt.tight_layout(); plt.show()

**What to notice:** RBF **`gamma`** sets each point's reach. **Small `gamma`** → wide influence →
smooth, near-linear boundaries (can underfit). **Large `gamma`** → tight, local influence → wiggly
boundaries that can wrap individual points (overfit). Together with `C`, it's the RBF SVM's main
tuning knob.

## Gotchas & tradeoffs

- **RBF has two knobs to tune** (`gamma` and `C`), and they interact — grid-search them together with
  cross-validation. Large `gamma` + large `C` overfits spectacularly.
- **Kernels need scaled features.** The RBF kernel depends on `‖x−x'‖`, so unscaled features distort
  every similarity — standardize first.
- **Kernel SVMs don't scale to huge `n`.** The Gram matrix is `O(n²)` memory and training is roughly
  `O(n²–n³)` — for millions of points use linear SVMs or approximate features (Nyström, random
  Fourier features).
- **Kernel choice is a modeling decision.** RBF is a strong default, but poly/linear/custom kernels
  encode different assumptions about similarity.

In [ ]:
# Kernel SVM cost: the Gram matrix is O(n^2) -> memory blows up with dataset size
for n in [1_000, 10_000, 100_000]:
    gram_gb = n * n * 8 / 1e9        # float64 Gram matrix
    print(f'n={n:>7}: Gram matrix = {n*n:,} entries = {gram_gb:.2f} GB')
print('\n-> this quadratic memory is why kernel SVMs are reserved for small/medium datasets')

**What to notice:** the kernel Gram matrix grows **quadratically** — 100k points already needs 80 GB
just to store pairwise kernels. This is the fundamental scaling limit of kernel SVMs and the reason
large-scale problems use linear SVMs or approximate kernel features instead.

## Key takeaways

- Kernels let a linear SVM learn **non-linear** boundaries via implicit feature maps.
- **RBF** is the go-to general-purpose kernel; **polynomial** and **linear** are alternatives.
- `C` controls margin softness; `gamma` controls RBF reach — tune both together (grid search).
- Large `gamma` or `C` overfits; always scale features before kernel SVMs.

---
## ✏️ Your turn

The cells below are **exercise scaffolds**: the concept is recapped, the code outline is set, and `# TODO(you)` marks what you fill in. Run the `assert` cell after each — it passes silently when your answer is right.

### Exercise 1 — The RBF kernel

The radial basis function kernel scores similarity by distance:

$$k(\mathbf{x}, \mathbf{z}) = \exp\!\big(-\gamma \, \lVert \mathbf{x} - \mathbf{z} \rVert^2\big)$$

Implement it. The checks verify the properties that make it a kernel — $k(\mathbf{x}, \mathbf{x}) = 1$, symmetry — and the $\gamma$ behavior from the section above: bigger $\gamma$ makes similarity die faster with distance.

In [ ]:
def rbf(x, z, gamma=1.0):
    """RBF (Gaussian) kernel between two vectors."""
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)

    # TODO(you): squared Euclidean distance between x and z
    sq_dist = ...

    # TODO(you): exp(-gamma * squared distance)
    return ...

In [ ]:
# Checks — run me
assert abs(rbf([1, 2], [1, 2]) - 1.0) < 1e-12, "a point is perfectly similar to itself"
assert abs(rbf([0, 0], [1, 0], gamma=1.0) - np.exp(-1)) < 1e-12, "unit distance, gamma=1 -> e^-1"
assert abs(rbf([1, 3], [2, 5]) - rbf([2, 5], [1, 3])) < 1e-15, "kernels are symmetric"
assert rbf([0, 0], [1, 0], gamma=10.0) < rbf([0, 0], [1, 0], gamma=0.1), \
    "bigger gamma -> similarity dies faster with distance"

# Edge cases
assert abs(rbf([0, 0], [100, 100], gamma=1.0)) < 1e-12, "far-apart points are (numerically) dissimilar"
assert abs(rbf([1, 2, 3], [1, 2, 3], gamma=5.0) - 1.0) < 1e-12, "self-similarity holds in higher dimensions too"
assert abs(rbf([5, 5], [5, 5], gamma=0.0) - 1.0) < 1e-12, "gamma=0 collapses the kernel to a constant 1 (no distance decay)"
assert abs(rbf([-1, -2], [3, 1]) - rbf([1, 2], [-3, -1])) < 1e-12, \
    "squared distance is invariant to negating both points, so the kernel is too"
print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def rbf(x, z, gamma=1.0):
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)
    sq_dist = np.sum((x - z) ** 2)
    return np.exp(-gamma * sq_dist)
```

</details>

### Exercise 2 — The kernel trick, verified

The quadratic kernel $k(\mathbf{x}, \mathbf{z}) = (\mathbf{x} \cdot \mathbf{z})^2$ secretly computes a dot product in 3D feature space:

$$\varphi(\mathbf{x}) = \big(x_1^2, \; \sqrt{2}\, x_1 x_2, \; x_2^2\big)
\qquad\Rightarrow\qquad
k(\mathbf{x}, \mathbf{z}) = \varphi(\mathbf{x}) \cdot \varphi(\mathbf{z})$$

Implement both sides and let the checks confirm they agree on random inputs — the kernel gets the 3D answer **without ever building the 3D vectors**.

In [ ]:
def poly2_kernel(x, z):
    """Quadratic kernel (x . z)^2 — never leaves 2D."""
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)

    # TODO(you): square of the ordinary dot product
    return ...


def phi(x):
    """The explicit 3D feature map (x1^2, sqrt(2) x1 x2, x2^2)."""
    x1, x2 = float(x[0]), float(x[1])

    # TODO(you): build the 3-vector
    return ...

In [ ]:
# Checks — run me
rng = np.random.default_rng(0)
for _ in range(5):
    x, z = rng.standard_normal(2), rng.standard_normal(2)
    assert abs(poly2_kernel(x, z) - np.dot(phi(x), phi(z))) < 1e-9, \
        "(x.z)^2 must equal the dot product in the explicit 3D feature space"

assert abs(poly2_kernel([1, 2], [3, 1]) - 25.0) < 1e-12, "(1*3 + 2*1)^2 = 25"

# Edge cases
assert abs(poly2_kernel([1, 0], [0, 1])) < 1e-12, "orthogonal vectors -> zero dot product -> kernel is 0"
assert abs(poly2_kernel([0, 0], [3, 4])) < 1e-12, "the zero vector has zero dot product with anything"
assert abs(poly2_kernel([2, -3], [-1, 4]) - poly2_kernel([-1, 4], [2, -3])) < 1e-12, "the kernel is symmetric"
assert np.allclose(phi([0, 0]), [0.0, 0.0, 0.0]), "phi of the zero vector is the zero vector"
print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def poly2_kernel(x, z):
    x = np.asarray(x, dtype=float)
    z = np.asarray(z, dtype=float)
    return float(np.dot(x, z) ** 2)


def phi(x):
    x1, x2 = float(x[0]), float(x[1])
    return np.array([x1 ** 2, np.sqrt(2) * x1 * x2, x2 ** 2])
```

</details>

---
## 🔬 Extra practice — DML `45_linear-kernel-function` and `21_pegasos-kernel-svm-implementation`

[Open-Deep-ML problem 45](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/45_linear-kernel-function)
and [problem 21](https://github.com/Open-Deep-ML/DML-OpenProblem/tree/main/questions/21_pegasos-kernel-svm-implementation)
train an SVM through a completely different route than the `sklearn.svm.SVC`
calls used earlier in this notebook. Instead of the usual quadratic-programming
/ SMO solver, **Pegasos** is sub-gradient descent applied directly to
per-sample dual coefficients, evaluated only through kernel calls — never an
explicit weight vector. Problem 45 is the tiny linear-kernel building block;
problem 21 is the full solver, usable with either that linear kernel or RBF.

### Extra practice 1 — DML `45_linear-kernel-function`

A one-line warm-up: implement the linear kernel as the ordinary dot product,
matching DML's exact function name `kernel_function`. It should agree with
this notebook's own `linear_kernel` from the first section.

In [ ]:
def kernel_function(x1, x2):
    """DML 45: linear kernel = dot product of two vectors."""
    x1 = np.asarray(x1, dtype=float)
    x2 = np.asarray(x2, dtype=float)

    # TODO(you): the linear kernel is just the ordinary dot product
    return ...

In [ ]:
# Checks — run me (DML's own published test cases, from tests.json)
assert kernel_function(np.array([1, 2, 3]), np.array([4, 5, 6])) == 32
assert kernel_function(np.array([0, 1, 2]), np.array([3, 4, 5])) == 14

# Edge cases: zero vector -> 0; orthogonal vectors -> 0; agrees with this
# notebook's own linear_kernel from the first section
assert kernel_function(np.array([0, 0, 0]), np.array([9, -3, 7])) == 0
assert kernel_function(np.array([1, 0]), np.array([0, 1])) == 0
assert kernel_function(np.array([-2, 3]), np.array([4, -1])) == linear_kernel(np.array([-2, 3]), np.array([4, -1]))
print("✅ Extra practice 1 (DML 45) passed")

<details>
<summary>💡 Show solution</summary>

```python
def kernel_function(x1, x2):
    x1 = np.asarray(x1, dtype=float)
    x2 = np.asarray(x2, dtype=float)
    return float(np.dot(x1, x2))
```

</details>

### Extra practice 2 — DML `21_pegasos-kernel-svm-implementation`

The Pegasos algorithm trains an SVM by sub-gradient descent directly on
**dual coefficients** $\alpha_i$ and a bias $b$, evaluated only through
kernel calls $K(x_i, x_j)$ — never an explicit weight vector, which is what
lets it work with nonlinear kernels like RBF. At step $t$ (learning rate
$\eta_t = 1/(\lambda t)$), for every sample $i$:

$$\text{decision}_i = \sum_j \alpha_j y_j K(x_j, x_i) + b$$

If $y_i \cdot \text{decision}_i < 1$ (the margin is violated), take a
sub-gradient step:

$$\alpha_i \mathrel{+}= \eta_t (y_i - \lambda \alpha_i), \qquad b \mathrel{+}= \eta_t y_i$$

**Note:** DML's version of Pegasos is deliberately **deterministic** — the
original algorithm samples one random point per step, but this variant loops
over *every* sample each iteration (no randomness), so results are exactly
reproducible.

In [ ]:
def pegasos_kernel_svm(data, labels, kernel='linear', lambda_val=0.01, iterations=100, sigma=1.0):
    """DML 21: deterministic (non-stochastic) Pegasos sub-gradient SVM solver,
    operating entirely through a kernel -- no explicit weight vector, just
    per-sample dual coefficients `alphas` and a bias `b`.
    """
    data = np.asarray(data, dtype=float)
    labels = np.asarray(labels, dtype=float)
    n_samples = len(data)
    alphas = np.zeros(n_samples)
    b = 0.0

    if kernel == 'linear':
        kernel_func = kernel_function
    elif kernel == 'rbf':
        # DML parameterizes the RBF kernel by sigma; this notebook's rbf_kernel
        # takes gamma, so convert: gamma = 1 / (2 * sigma^2)
        kernel_func = lambda x, z: rbf_kernel(x, z, gamma=1.0 / (2 * sigma ** 2))
    else:
        raise ValueError(f"unknown kernel: {kernel}")

    for t in range(1, iterations + 1):
        eta = 1.0 / (lambda_val * t)
        for i in range(n_samples):
            # TODO(you): decision_i = sum_j alphas[j] * labels[j] * K(data[j], data[i]) + b
            decision = ...

            # TODO(you): if the margin is violated (labels[i] * decision < 1), take
            # the sub-gradient step:
            #   alphas[i] += eta * (labels[i] - lambda_val * alphas[i])
            #   b        += eta * labels[i]
            ...

    return np.round(alphas, 4).tolist(), float(np.round(b, 4))

In [ ]:
# Checks — run me (DML's own published test cases, from tests.json)
data = np.array([[1, 2], [2, 3], [3, 1], [4, 1]])
labels = np.array([1, 1, -1, -1])

alphas_lin, b_lin = pegasos_kernel_svm(data, labels, kernel='linear', lambda_val=0.01, iterations=100)
assert alphas_lin == [100.0, 0.0, -100.0, -100.0]
assert abs(b_lin - (-937.4755)) < 1e-4

alphas_rbf, b_rbf = pegasos_kernel_svm(data, labels, kernel='rbf', lambda_val=0.01, iterations=100, sigma=0.5)
assert alphas_rbf == [100.0, 99.0, -100.0, -100.0]
assert abs(b_rbf - (-115.0)) < 1e-4

print("✅ Extra practice 2 (DML 21) passed")

<details>
<summary>💡 Show solution</summary>

```python
def pegasos_kernel_svm(data, labels, kernel='linear', lambda_val=0.01, iterations=100, sigma=1.0):
    data = np.asarray(data, dtype=float)
    labels = np.asarray(labels, dtype=float)
    n_samples = len(data)
    alphas = np.zeros(n_samples)
    b = 0.0

    if kernel == 'linear':
        kernel_func = kernel_function
    elif kernel == 'rbf':
        kernel_func = lambda x, z: rbf_kernel(x, z, gamma=1.0 / (2 * sigma ** 2))
    else:
        raise ValueError(f"unknown kernel: {kernel}")

    for t in range(1, iterations + 1):
        eta = 1.0 / (lambda_val * t)
        for i in range(n_samples):
            decision = sum(alphas[j] * labels[j] * kernel_func(data[j], data[i])
                            for j in range(n_samples)) + b
            if labels[i] * decision < 1:
                alphas[i] += eta * (labels[i] - lambda_val * alphas[i])
                b += eta * labels[i]

    return np.round(alphas, 4).tolist(), float(np.round(b, 4))
```

</details>